# Example 11.1 / 3: Recognize Common Quadratic Patterns

Companion notebook for **11.1 Quadratic Models** in the AMPL lab Documentation, section "Recognize Common Quadratic Patterns". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Recognize Common Quadratic Patterns

Many applications reduce to a small number of quadratic structures. Classify the expression
before choosing a solver, and keep the data and decision variables distinct:

| Pattern | Quadratic structure | Convexity condition |
|---|---|---|
| ordinary least squares | $\min_\theta \lVert y-X\theta\rVert_2^2$ | $2X^TX$ is positive semidefinite |
| ridge regularization | add $\lambda\lVert\theta\rVert_2^2$ with $\lambda\ge0$ | preserves convexity and can add strict curvature |
| mean-variance portfolio | variance $x^T\Sigma x$ in an objective or risk bound | covariance matrix $\Sigma$ is positive semidefinite |
| quadratic penalty | weighted squared deviations from targets | nonnegative weights give a convex penalty |

In least squares, $X$ and $y$ are data while the fitted coefficients are variables. The Hessian
$2X^TX$ is always positive semidefinite, although rank deficiency can leave several coefficient
vectors with the same fitted values. In a portfolio model, estimated covariance data should be
checked for symmetry and positive semidefiniteness; a noisy or incorrectly assembled matrix can
change a convex model into an indefinite one.

The complete regression workflow belongs with Optimization Models in ML Pipelines (AMPL lab documentation, section 12.8.1),
and complete portfolio models are in the [Models](/models) collection.

Ordinary least squares and ridge regularization on the same three observations
$(x,y)=(1,1),(2,2),(3,2)$: the fitted slope is $\theta=11/14\approx0.785714$ without the ridge
term and $11/15\approx0.733333$ with $\lambda=1$, so the penalty visibly shrinks the coefficient.

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set OBS;            # observations
param xv {OBS};     # regressor value per observation
param yv {OBS};     # observed response
param lambda >= 0;  # ridge weight; 0 gives ordinary least squares

var Theta;  # fitted slope, no intercept

minimize Fit:  # squared residuals plus ridge penalty
    sum {i in OBS} (yv[i] - Theta * xv[i])^2 + lambda * Theta^2;

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["OBS"] = [1, 2, 3]
ampl.param["xv"] = {1: 1, 2: 2, 3: 3}
ampl.param["yv"] = {1: 1, 2: 2, 3: 2}
ampl.param["lambda"] = 0

In [5]:
# run.py
for lam in (0, 1):
    ampl.param["lambda"] = lam
    ampl.solve(solver="highs", verbose=False)
    print(lam, ampl.solve_result, round(ampl.var["Theta"].value(), 6),
          round(ampl.obj["Fit"].value(), 6))

0 solved 0.785714 0.357143
1 solved 0.733333 0.933333
